# DiFuMo MLP/GAT Saved-Artifact Comparison

Reads saved JSON/CSV artifacts from Google Drive and makes comparison tables/plots. This notebook does not train models and does not load checkpoints into GPU memory.

In [ ]:
import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
!pip -q install pandas matplotlib seaborn

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os
import pandas as pd

DRIVE_ROOTS = {
    'difumo_gat': Path('/content/drive/MyDrive/neurovlm_difumo_gat'),
    'difumo_mlp': Path('/content/drive/MyDrive/neurovlm_difumo_mlp'),
}
RUN_ROOTS = {
    'difumo_gat': DRIVE_ROOTS['difumo_gat'] / 'runs_difumo_gat',
    'difumo_mlp': DRIVE_ROOTS['difumo_mlp'] / 'runs_difumo_mlp',
}
COMPARISON_FILES = {
    'difumo_gat': RUN_ROOTS['difumo_gat'] / 'difumo_gat_comparison.csv',
    'difumo_mlp': RUN_ROOTS['difumo_mlp'] / 'difumo_mlp_comparison.csv',
}
COMPARE_ROOT = Path('/content/drive/MyDrive/neurovlm_difumo_comparison')
COMPARE_ROOT.mkdir(parents=True, exist_ok=True)
OUT_FILE = COMPARE_ROOT / 'difumo_saved_artifact_summary.csv'
PLOT_FILE = COMPARE_ROOT / 'difumo_saved_artifact_semantic_metrics.png'
for name, root in RUN_ROOTS.items():
    print(f'{name} run root:', root)
print('Output summary:', OUT_FILE)


In [ ]:
def load_json(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception:
        return {}

def first_existing(*values):
    for value in values:
        if value is not None:
            return value
    return None

rows = []
for family, run_root in RUN_ROOTS.items():
    eval_paths = sorted(run_root.glob('*/*/eval_results.json'))
    if not eval_paths and (run_root.parent / 'runs').exists():
        eval_paths = sorted((run_root.parent / 'runs').glob('*/*/eval_results.json'))
    for eval_path in eval_paths:
        run_dir = eval_path.parent
        batch = run_dir.parent.name
        run_name = run_dir.name
        config = load_json(run_dir / 'config.json')
        graph = load_json(run_dir / 'graph_stats.json')
        metrics_payload = load_json(eval_path)
        test = metrics_payload.get('test', {})
        val = metrics_payload.get('val', {})
        manifest = load_json(run_dir / 'artifacts_manifest.json')
        sem = load_json(run_dir / 'main_comparison_summary_row.json')
        rows.append({
            'family': family,
            'batch': batch,
            'run': run_name,
            'model': first_existing(sem.get('model'), config.get('model')),
            'coefficient_source': config.get('coeff_source'),
            'ale_kernel_fwhm_mm': config.get('ale_kernel_fwhm_mm') if config.get('coeff_source') == 'ale_coordinates' else None,
            'graph_type': first_existing(sem.get('graph_type'), config.get('graph_type') if config.get('model') == 'gat' else 'none'),
            'conv': config.get('conv') if config.get('model') == 'gat' else '',
            'hidden': config.get('hidden'),
            'heads': config.get('heads'),
            'layers': config.get('layers'),
            'batch_size': config.get('batch_size'),
            'lr_gat': config.get('lr_gat'),
            'lr_proj': config.get('lr_proj'),
            'temperature': config.get('temperature'),
            'exact_auc': first_existing(sem.get('exact_pmid_paper_recall_curve_auc'), test.get('mean_auc')),
            'exact_r@1': first_existing(sem.get('exact_pmid_recall@1'), test.get('mean_recall@1')),
            'exact_r@5': first_existing(sem.get('exact_pmid_recall@5'), test.get('mean_recall@5')),
            'exact_r@10': first_existing(sem.get('exact_pmid_recall@10'), test.get('mean_recall@10')),
            'exact_r@50': first_existing(sem.get('exact_pmid_recall@50'), test.get('mean_recall@50')),
            'exact_mrr': first_existing(sem.get('exact_pmid_mrr'), test.get('mean_mrr')),
            'exact_median_rank': first_existing(sem.get('exact_pmid_median_rank'), test.get('mean_median_rank')),
            'test_random_r@10': test.get('mean_random_recall@10'),
            'val_auc': val.get('mean_auc'),
            'network_accuracy': sem.get('network_accuracy'),
            'network_top_2_accuracy': sem.get('network_top_2_accuracy'),
            'network_macro_auc': sem.get('network_macro_auc'),
            'mesh_recall@5': sem.get('mesh_recall@5'),
            'mesh_recall@10': sem.get('mesh_recall@10'),
            'mesh_map': sem.get('mesh_map'),
            'mesh_mrr': sem.get('mesh_mrr'),
            'mesh_median_best_true_term_rank': sem.get('mesh_median_best_true_term_rank'),
            'semantic_recall@10': sem.get('semantic_recall@10'),
            'semantic_recall@50': sem.get('semantic_recall@50'),
            'semantic_mrr': sem.get('semantic_mrr'),
            'semantic_paper_style_recall_curve_auc': sem.get('semantic_paper_style_recall_curve_auc'),
            'n_edges': graph.get('number_of_edges'),
            'avg_degree': first_existing(graph.get('average_degree_directed'), graph.get('average_degree')),
            'connected_components': graph.get('connected_components'),
            'edge_weight_mean': graph.get('edge_weight_mean'),
            'edge_weight_std': graph.get('edge_weight_std'),
            'brain_parameters': config.get('brain_parameters'),
            'peak_vram': first_existing(sem.get('peak_vram'), manifest.get('comparison_row_payload', {}).get('peak_vram')),
            'training_time_per_epoch': first_existing(sem.get('training_time_per_epoch'), manifest.get('comparison_row_payload', {}).get('training_time_per_epoch')),
            'run_dir': str(run_dir),
            'best_checkpoint': manifest.get('best_checkpoint', str(run_dir / f'checkpoints_{family}' / 'best_difumo_gat.pt')),
            'config_path': str(run_dir / 'config.json'),
            'eval_results_path': str(eval_path),
            'main_comparison_summary_row': str(run_dir / 'main_comparison_summary_row.csv'),
        })

summary = pd.DataFrame(rows)
if len(summary):
    sort_col = 'mesh_recall@10' if summary['mesh_recall@10'].notna().any() else 'exact_auc'
    summary = summary.sort_values(sort_col, ascending=False, na_position='last').reset_index(drop=True)
display(summary)
summary.to_csv(OUT_FILE, index=False)
print('Saved:', OUT_FILE)


In [ ]:
import matplotlib.pyplot as plt

if len(summary):
    plot_df = summary.copy()
    labels = plot_df['family'] + ' | ' + plot_df['run'] + ' | ' + plot_df['coefficient_source'].fillna('') + ' | ' + plot_df['graph_type'].fillna('')
    plot_cols = [c for c in ['network_accuracy', 'mesh_recall@10', 'semantic_recall@10', 'exact_r@10'] if c in plot_df.columns]
    fig, ax = plt.subplots(figsize=(11, max(4, 0.38 * len(plot_df))))
    if plot_cols:
        plot_df.assign(label=labels).set_index('label')[plot_cols].plot(kind='barh', ax=ax)
        ax.set_xlabel('score')
        ax.set_title('DiFuMo Semantic Metrics and Exact PMID Diagnostic')
        ax.invert_yaxis()
        ax.legend(loc='best')
    fig.tight_layout()
    display(fig)
    fig.savefig(PLOT_FILE, dpi=160)
    plt.close(fig)


In [ ]:
if len(summary):
    metric_cols = [
        'family', 'run', 'model', 'coefficient_source', 'graph_type',
        'network_accuracy', 'network_top_2_accuracy', 'network_macro_auc',
        'mesh_recall@5', 'mesh_recall@10', 'mesh_map', 'mesh_mrr',
        'semantic_recall@10', 'semantic_recall@50', 'semantic_mrr',
        'exact_auc', 'exact_r@1', 'exact_r@5', 'exact_r@10', 'exact_r@50', 'exact_mrr', 'exact_median_rank',
        'peak_vram', 'training_time_per_epoch', 'run_dir',
    ]
    display(summary[[c for c in metric_cols if c in summary.columns]])


## Interpretation Checklist

- Exact PMID retrieval is strict and diagnostic; do not treat low exact recall as the only failure mode.
- Raw GAT > raw MLP on semantic metrics: graph structure helps on the original DiFuMo coefficients.
- ALE-smoothed MLP > raw MLP: ALE smoothing improves the coefficient representation.
- ALE-smoothed GAT > ALE-smoothed MLP: graph structure helps after smoothing.
- Real-HCP GAT > coactivation/spatial GAT: graph quality matters.
- Shuffled/no-edge GAT approximately real-graph GAT: the graph is not adding much.
